# OMRモデルの推論

In [1]:
import yaml
import torch
from yolov3.models.yolo import (
    BaseModel,
)  # notebookの場合パスが通らないのでこのようにする
from src.domain.model import OMRModel
from src.domain.dataloader import CustomDataset, custom_collate_fn
from src.domain.loss import CustomLoss
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim

from torchvision import transforms
from PIL import Image
from src.utils import non_max_suppression
import cv2

/home/docker/.cache/pypoetry/virtualenvs/repo-uZbbGesQ-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
model_dir = "../models/"
config_path = model_dir + "config/omr_yolov5s.yaml"  #'yolov3/models/yolov5s.yaml'
model_path = model_dir + "pre_trained/yolov5s.pt"

data_dir = "../data/"
hyp_path = data_dir + "hyps/hyp.scratch-low.yaml"
img_dir = data_dir + "dense/images/"
annotation_dir = data_dir + "dense/labels_mapping/"

EXP_NAME = "250417_complete"
fine_tuned_path = model_dir + f"fine_tuned/{EXP_NAME}/omr_yolov5s.pth"

ratio_wh = 1
RESIZE_SIZE_W = 1280
RESIZE_SIZE_H = int(ratio_wh * RESIZE_SIZE_W)
print(RESIZE_SIZE_H, RESIZE_SIZE_W)

1280 1280


In [39]:
model = OMRModel(config_path)
model.load_state_dict(torch.load(fine_tuned_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 推論モード
model.eval()

image_path = data_dir + "dense/images/1.png"
im0 = cv2.imread(image_path)
image = Image.open(image_path).convert("RGB")
transform = transforms.Compose(
    [
        transforms.Resize((RESIZE_SIZE_W, RESIZE_SIZE_H)),
        transforms.ToTensor(),
    ]
)
image = transform(image).unsqueeze(0)
# 推論
with torch.no_grad():  # メモリ効率を向上させるためにtorch.no_gradを使用
    input = image.to(device)
    pred = model(input)

# 推論結果を表示または処理するコードを追加
print(pred)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  yolov3.models.common.Conv               [3, 32, 6, 2, 2]              
  1                -1  1     18560  yolov3.models.common.Conv               [32, 64, 3, 2]                
  2                -1  1     18816  yolov3.models.common.C3                 [64, 64, 1]                   
  3                -1  1     73984  yolov3.models.common.Conv               [64, 128, 3, 2]               
  4                -1  2    115712  yolov3.models.common.C3                 [128, 128, 2]                 
  5                -1  1    295424  yolov3.models.common.Conv               [128, 256, 3, 2]              
  6                -1  3    625152  yolov3.models.common.C3                 [256, 256, 3]                 
  7                -1  1   1180672  yolov3.models.common.Conv               [256, 512, 3, 2]              
  8                -1  1   1182720  

omr_YOLOv3s summary: 214 layers, 7456543 parameters, 7456543 gradients, 17.3 GFLOPs



(tensor([[[ 7.44790e+00,  2.20750e-01,  9.05646e+00,  ...,  2.34288e-01,  2.09777e-01,  1.67529e-01],
         [ 1.08654e+01,  1.68712e-01,  1.01981e+01,  ...,  2.09808e-01,  1.97806e-01,  1.98280e-01],
         [ 1.99520e+01, -1.83766e-01,  1.07967e+01,  ...,  1.69314e-01,  1.63191e-01,  1.57333e-01],
         ...,
         [ 1.19820e+03,  1.23932e+03,  2.74818e+01,  ...,  4.44300e-04,  1.48536e-06,  2.70475e-02],
         [ 1.22818e+03,  1.23945e+03,  1.93801e+01,  ...,  3.26603e-04,  1.65229e-06,  1.67988e-02],
         [ 1.26315e+03,  1.26849e+03,  2.17661e+01,  ...,  1.11265e-04,  1.52099e-06,  1.42298e-02]]], device='cuda:0'), [tensor([[[[[ 9.22219e-01, -1.02633e+00,  2.76640e-01,  ..., -1.18425e+00, -1.32627e+00, -1.60324e+00],
           [-2.85569e-01, -1.04314e+00,  4.20085e-01,  ..., -1.32608e+00, -1.40006e+00, -1.39708e+00],
           [-1.19883e-02, -1.16084e+00,  4.93648e-01,  ..., -1.59050e+00, -1.63468e+00, -1.67821e+00],
           ...,
           [-4.43861e-01, -1.2254

In [40]:
print(isinstance(pred, (list, tuple)))
print("len(pred)", len(pred))
print("pred[0]のshape", pred[0].shape)  # nc+npitch+xyhw+conf
print("len(pred[1])", len(pred[1]))
print(pred[1][0].shape)
print(pred[1][1].shape)
print(pred[1][2].shape)
print(pred[0][0][0])

print("conf max", pred[0][..., 4].max())
print("x max", pred[0][..., 0].min())
print(pred[0][pred[0][..., 4] > 0.2][:, 4])

True
len(pred) 2
pred[0]のshape torch.Size([1, 100800, 167])
len(pred[1]) 3
torch.Size([1, 3, 160, 160, 167])
torch.Size([1, 3, 80, 80, 167])
torch.Size([1, 3, 40, 40, 167])
tensor([7.44790e+00, 2.20750e-01, 9.05646e+00, 5.30982e+00, 5.97745e-02, 4.11324e-03, 3.13057e-03, 3.08238e-03, 4.31189e-03, 4.88493e-03, 4.09593e-03, 2.49010e-01, 1.73674e-01, 1.88024e-01, 1.93087e-01, 3.32936e-03, 3.65277e-03, 3.07167e-03, 4.34240e-03, 3.55000e-03, 2.84282e-03, 5.44763e-03, 4.50817e-03, 2.88796e-03,
        4.04814e-03, 3.83295e-03, 3.83238e-03, 3.38001e-03, 5.71535e-03, 5.49981e-03, 4.46595e-01, 3.82087e-03, 4.04248e-01, 5.07436e-03, 2.88080e-01, 5.35562e-03, 2.89551e-01, 7.00982e-03, 3.73889e-01, 4.66726e-03, 3.70802e-01, 2.47298e-03, 9.09970e-02, 3.18981e-03, 7.11982e-02, 5.44365e-03, 4.22562e-03, 5.40416e-03,
        4.51575e-03, 3.77290e-03, 4.67447e-03, 2.75302e-03, 4.84647e-03, 4.06390e-03, 2.91092e-03, 3.34294e-03, 4.75811e-03, 3.79005e-03, 3.65861e-03, 5.22278e-03, 3.95188e-03, 3.44485e-0

In [41]:
print("w", im0.shape[1])
print("h", im0.shape[0])

w 1960
h 2772


In [42]:
output = non_max_suppression(
    pred,
    conf_thres=0.17,
    w=im0.shape[1] / RESIZE_SIZE_W,
    h=im0.shape[0] / RESIZE_SIZE_H,
)
print(output[0].shape)
print(output[0])
print(output[0].cpu().numpy())

x tensor([[5.37265e+02, 4.06777e+01, 1.41797e+01,  ..., 3.35400e-01, 3.12535e-01, 2.95036e-01],
        [1.05921e+03, 3.95964e+01, 1.91791e+01,  ..., 3.88809e-01, 3.67192e-01, 3.46416e-01],
        [5.38051e+02, 4.75281e+01, 1.57659e+01,  ..., 3.37781e-01, 3.23262e-01, 3.05575e-01],
        ...,
        [1.13573e+02, 1.21021e+03, 1.30097e+01,  ..., 1.35656e-05, 1.20350e-06, 3.94668e-04],
        [1.73308e+02, 1.22343e+03, 2.66436e+01,  ..., 1.24137e-05, 1.23247e-06, 7.16458e-04],
        [1.73433e+02, 1.22226e+03, 2.59239e+01,  ..., 1.38579e-05, 1.16940e-06, 2.33194e-04]], device='cuda:0')
x tensor([[5.37265e+02, 4.06777e+01, 1.41797e+01,  ..., 6.39945e-02, 5.96320e-02, 5.62931e-02],
        [1.05921e+03, 3.95964e+01, 1.91791e+01,  ..., 8.40355e-02, 7.93634e-02, 7.48730e-02],
        [5.38051e+02, 4.75281e+01, 1.57659e+01,  ..., 7.25053e-02, 6.93887e-02, 6.55922e-02],
        ...,
        [1.13573e+02, 1.21021e+03, 1.30097e+01,  ..., 3.70895e-06, 3.29046e-07, 1.07906e-04],
        [1.7

In [43]:
from ultralytics.utils.plotting import Annotator, colors
from yolov3.utils.general import (
    cv2,
    scale_boxes,
)

save_crop = False  # save cropped prediction boxes
line_thickness = 3  # bounding box thickness (pixels)
save_txt = False  # save results to *.txt
view_img = True  # show results
nosave = False  # do not save images/videos
hide_labels = False  # hide labels
hide_conf = False  # hide confidences

names = model.names
# im0 = im0.to(device)
im = image.to(device)
output_path = "../data/output/dense/detection/"

print("読み込み完了")
# Process predictions
for i, det in enumerate(output):  # per image
    print("処理開始")
    # det = det.cpu()
    gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh
    imc = im0.copy() if save_crop else im0  # for save_crop
    annotator = Annotator(im0, line_width=line_thickness, example=str(names))
    if len(det):
        # Rescale boxes from img_size to im0 size
        # det[:, :4] = scale_boxes(im.shape[2:], det[:, :4], im0.shape).round()
        print("det", det)

        # Print results
        for c in det[:, 5].unique():
            n = (det[:, 5] == c).sum()  # detections per class
            # s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string

        # Write results
        for *xyxy, conf, cls in reversed(det):
            # print("annotator")
            # if save_txt:  # Write to file
            #     xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()  # normalized xywh
            #     line = (cls, *xywh, conf) if save_conf else (cls, *xywh)  # label format
            #     with open(f"{txt_path}.txt", "a") as f:
            #         f.write(("%g " * len(line)).rstrip() % line + "\n")

            c = int(cls.cpu())  # integer class
            label = (
                None
                if hide_labels
                else (names[c] if hide_conf else f"{names[c]} {conf:.2f}")
            )
            annotator.box_label(xyxy, label, color=colors(c, True))

    result_image = annotator.result()
    # cv2.imshow("result", result_image)
    cv2.imwrite(output_path + f"1_{EXP_NAME}.png", result_image)

読み込み完了
処理開始
det tensor([[1.06952e+02, 1.06010e+02, 1.48419e+02, 2.15693e+02, 2.31990e-01, 6.00000e+00],
        [1.06393e+02, 2.59275e+03, 1.48661e+02, 2.64589e+03, 1.90946e-01, 9.00000e+00],
        [1.64062e+02, 1.33220e+02, 1.78605e+02, 1.73451e+02, 1.84596e-01, 6.80000e+01],
        [1.06920e+02, 1.08642e+03, 1.52539e+02, 1.22616e+03, 1.84411e-01, 6.00000e+00],
        [2.45833e+02, 2.96438e+02, 2.90445e+02, 4.27116e+02, 1.84184e-01, 6.00000e+00],
        [1.06416e+02, 2.56265e+03, 1.48252e+02, 2.61335e+03, 1.82632e-01, 9.00000e+00],
        [1.06146e+02, 1.26895e+02, 1.48002e+02, 1.81898e+02, 1.80343e-01, 9.00000e+00],
        [1.05747e+02, 4.89167e+02, 1.50827e+02, 6.23739e+02, 1.78729e-01, 6.00000e+00],
        [1.07002e+02, 8.83358e+02, 1.52056e+02, 1.02359e+03, 1.77032e-01, 6.00000e+00],
        [2.44978e+02, 2.60204e+03, 2.85776e+02, 2.69693e+03, 1.75545e-01, 6.00000e+00],
        [1.07013e+02, 6.77199e+02, 1.52262e+02, 8.21842e+02, 1.74924e-01, 6.00000e+00],
        [1.06960